# NumPy

NumPy is an open-source library for the Python programming language that adds support for large, multi-dimensional arrays and matrices, along with a large collection of mathematical and statistical functions to operate on these arrays.

Mathematical operations on NumPy's `ndarray` objects are up to 50x faster than iterating over native Python lists using loops.

- The NumPy array (`ndarray`) is the central object of the NumPy package.
- A one-dimensional array can be thought of as a vector, a two-dimensional array as a matrix, and a three-dimensional array as a tensor.
- Within an array, all elements must share the same data type.

## Learning Objectives

By the end of this notebook, you will be able to:

- Create and inspect NumPy arrays using multiple construction methods
- Understand and control array data types, and reason about their memory implications
- Understand array shape, reshaping, and the distinction between 1D and 2D arrays
- Explain broadcasting and apply it to perform arithmetic between arrays of different shapes
- Index and slice arrays using NumPy's axis-based syntax, including boolean indexing
- Use `np.where()` for conditional selection and replacement
- Compare arrays correctly, including floating-point comparisons with `np.allclose()`
- Apply statistical methods and universal functions to arrays
- Generate reproducible random data using a seeded generator
- Save and load arrays to and from disk

## Defining Arrays

### The `np.array()` function

Pass a Python list to `np.array()` to create a one-dimensional array (a vector).

In [1]:
# importing numpy and aliasing it as np
import numpy as np    # np is the convention

In [2]:
# Daily temperatures (Celsius) recorded at a weather station over one week
temps = np.array([28.5, 31.2, 30.0, 27.8, 29.4, 33.1, 32.6])
temps

array([28.5, 31.2, 30. , 27.8, 29.4, 33.1, 32.6])

In [3]:
temps.shape    # a 1D array of 7 elements; shape is a tuple with one value

(7,)

To create a 2D array, pass a list of lists. Each inner list becomes a row.

In [4]:
# Exam scores for 4 students across 3 exams
scores = np.array([
    [85, 90, 78],
    [92, 88, 95],
    [70, 65, 72],
    [88, 91, 84]
])
scores

array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84]])

In [5]:
scores.shape    # 4 rows (students), 3 columns (exam scores)

(4, 3)

### `np.arange()`

Behaves like Python's built-in `range()` but returns a NumPy array instead of a range object.

In [6]:
np.arange(10)           # 0 up to (not including) 10

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [7]:
np.arange(1, 6)         # 1 up to (not including) 6

array([1, 2, 3, 4, 5])

In [8]:
np.arange(0, 1, 0.1)    # start, stop, step - works with floats

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

### `np.zeros()`, `np.ones()`, `np.full()`

In [9]:
np.zeros(5)              # 1D array of five zeros

array([0., 0., 0., 0., 0.])

In [10]:
np.zeros((3, 4))         # 3x4 matrix of zeros; pass shape as a tuple

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [11]:
np.ones((2, 3))

array([[1., 1., 1.],
       [1., 1., 1.]])

In [12]:
np.full(shape=(3, 3), fill_value=7)    # every element is 7

array([[7, 7, 7],
       [7, 7, 7],
       [7, 7, 7]])

### `np.zeros_like()`, `np.ones_like()`, `np.full_like()`

These create a new array with the same shape (and optionally dtype) as an existing array.

In [13]:
# Useful when you need a blank slate with the same shape as an existing array
np.zeros_like(scores)

array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 0],
       [0, 0, 0]])

In [14]:
np.full_like(scores, fill_value=100)    # a perfect-score reference matrix

array([[100, 100, 100],
       [100, 100, 100],
       [100, 100, 100],
       [100, 100, 100]])

### Creating identity matrix using `np.identity()`

In [15]:
# Identity matrix: ones on the main diagonal, zeros elsewhere
np.identity(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

## Data Types

Every NumPy array has a single dtype that applies to all its elements, meaning that an array can not hold elements of mixed data types. The default data type for arrays created from Python floats is `float64`. The default for arrays created from Python integers is `int64` on most systems.

Common dtypes:

| dtype     | Description                | Bytes per element |
|-----------|----------------------------|-------------------|
| `int32`   | 32-bit signed integer      | 4                 |
| `int64`   | 64-bit signed integer      | 8                 |
| `float32` | 32-bit floating point      | 4                 |
| `float64` | 64-bit floating point      | 8                 |
| `bool`    | Boolean (True/False)       | 1                 |
| `str_`    | Fixed-width Unicode string | varies            |

Inspect the dtype of any array with the `.dtype` attribute.

In [16]:
print(scores.dtype)    # int64: all values were Python integers
print(temps.dtype)    # float64: values had decimal points

int64
float64


### Casting with `.astype()`

In [17]:
# Convert integer scores to float - useful before division to avoid integer truncation
scores_float = scores.astype(np.float64)
scores_float.dtype

dtype('float64')

In [18]:
# Convert to bool: zero becomes False, everything else becomes True
passing = scores.astype(bool)
passing

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True]])

### Why dtype matters: memory

Choosing a dtype that is wider than necessary wastes memory. This becomes significant with large arrays.

In [19]:
n = 1_000_000    # one million elements

arr_int64   = np.ones(n, dtype=np.int64)
arr_int32   = np.ones(n, dtype=np.int32)
arr_float64 = np.ones(n, dtype=np.float64)
arr_float32 = np.ones(n, dtype=np.float32)

print(f"int64   : {arr_int64.nbytes:>10,} bytes")
print(f"int32   : {arr_int32.nbytes:>10,} bytes")
print(f"float64 : {arr_float64.nbytes:>10,} bytes")
print(f"float32 : {arr_float32.nbytes:>10,} bytes")

int64   :  8,000,000 bytes
int32   :  4,000,000 bytes
float64 :  8,000,000 bytes
float32 :  4,000,000 bytes


The memory usage of the `scores` matrix defined earlier:

In [20]:
# The .nbytes attribute works on any array
print(scores)
print(f"scores uses {scores.nbytes} bytes ({scores.dtype})")

[[85 90 78]
 [92 88 95]
 [70 65 72]
 [88 91 84]]
scores uses 96 bytes (int64)


## Array Shape

### Inspecting shape

`.shape` returns a tuple where each element is the size of that dimension.

In [21]:
print(temps.shape)     # (7,)   - one dimension of size 7
print(scores.shape)    # (4, 3) - two dimensions

(7,)
(4, 3)


### Reshaping

In [22]:
# 12 hourly sensor readings taken over one day
readings = np.arange(1, 13)
readings

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [23]:
readings.reshape((3, 4))    # reshape into 3 blocks of 4 hours each

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

In [24]:
# -1 tells NumPy to calculate that dimension automatically
readings.reshape((2, -1))    # 2 rows, NumPy figures out 6 columns

array([[ 1,  2,  3,  4,  5,  6],
       [ 7,  8,  9, 10, 11, 12]])

In [25]:
# Reshaping does not change the underlying data
by_hour = readings.reshape((4, 3))
print(by_hour)
print("Shape:", by_hour.shape)

[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]
Shape: (4, 3)


A 1D array and a '2D array with one row' are not the same thing, even if they hold the same values:

In [26]:
a = np.array([1, 2, 3])    # A 1D array
b = a.reshape((1, 3))    # 2D matrix with one row (row vector)
c = a.reshape((3, 1))    # 2D matrix with one column (column vector)

print("a.shape:", a.shape)    # (3,)
print("b.shape:", b.shape)    # (1, 3)
print("c.shape:", c.shape)    # (3, 1)

a.shape: (3,)
b.shape: (1, 3)
c.shape: (3, 1)


### Transpose

In [27]:
print("Original shape:", scores.shape)
scores

Original shape: (4, 3)


array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84]])

In [28]:
# Rows become columns and columns become rows
print("Transposed shape:", scores.T.shape)
scores.T

Transposed shape: (3, 4)


array([[85, 92, 70, 88],
       [90, 88, 65, 91],
       [78, 95, 72, 84]])

## Arithmetic with NumPy Arrays

### Arithmetic with scalars

The scalar is broadcast to every element.

In [29]:
temps

array([28.5, 31.2, 30. , 27.8, 29.4, 33.1, 32.6])

In [30]:
temps + 5      # shift all readings up by 5 degrees

array([33.5, 36.2, 35. , 32.8, 34.4, 38.1, 37.6])

In [31]:
temps ** 2

array([ 812.25,  973.44,  900.  ,  772.84,  864.36, 1095.61, 1062.76])

### Element-wise arithmetic between arrays of the same shape

In [32]:
# Two weeks of temperature readings at two stations
week1 = np.array([28.5, 31.2, 30.0, 27.8, 29.4, 33.1, 32.6])
week2 = np.array([26.0, 29.5, 31.4, 28.9, 30.2, 31.8, 30.0])

week2 - week1    # day-by-day difference between stations

array([-2.5, -1.7,  1.4,  1.1,  0.8, -1.3, -2.6])

In [33]:
(week1 + week2) / 2    # element-wise average

array([27.25, 30.35, 30.7 , 28.35, 29.8 , 32.45, 31.3 ])

### Broadcasting

Broadcasting is NumPy's mechanism for performing arithmetic between arrays that do not have identical shapes. Instead of requiring you to manually replicate data, NumPy virtually stretches the smaller array to match the larger one.

**The rule**: two dimensions are compatible if they are equal, or if one of them is 1. NumPy aligns dimensions from the trailing axis (right to left) and applies the rule to each pair.

```
scores  shape: (4, 3)
bonus   shape: (3, )   <-- treated as (1, 3), stretched to (4, 3)
result  shape: (4, 3)
```

In [34]:
scores

array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84]])

In [35]:
# Add a different bonus to each exam (column), applied to all students (rows)
exam_bonus = np.array([2, 0, 3])    # shape (3, )
print(exam_bonus)
print(exam_bonus.shape)    # a tuple containing a single element

[2 0 3]
(3,)


In [36]:
scores + exam_bonus    # bonus broadcasts across all 4 rows

array([[87, 90, 81],
       [94, 88, 98],
       [72, 65, 75],
       [90, 91, 87]])

In [37]:
# Add a different bonus to each student (row), applied to all exams (columns)
student_bonus = np.array([5, 0, 3, 1]).reshape((4, 1))    # shape (4, 1)
scores + student_bonus    # broadcasts across all 3 columns

array([[90, 95, 83],
       [92, 88, 95],
       [73, 68, 75],
       [89, 92, 85]])

In [38]:
# Scalar broadcast: the scalar is applied to every element
scores * 1.1    # scale all scores by 10%

array([[ 93.5,  99. ,  85.8],
       [101.2,  96.8, 104.5],
       [ 77. ,  71.5,  79.2],
       [ 96.8, 100.1,  92.4]])

When shapes are incompatible, NumPy raises a `ValueError`. Uncomment the line below to see it:

In [39]:
incompatible = np.array([1, 2, 3, 4])    # shape (4,)
# UNCOMMENT TO SEE THE ERROR:
# scores + incompatible    # (4, 3) and (4,) - trailing dimensions 3 and 4 are not compatible

## Indexing and Slicing

Indexing rules:
- Indices start at 0
- Commas separate axes: `arr[row, col]`
- Colons mean "through": `arr[0:3]` is rows 0, 1, 2
- Negative indices count from the end: `arr[-1]` is the last row
- A blank before or after a colon means "the rest": `arr[2:]`, `arr[:3]`, `arr[:]`

In [40]:
# Use a 5x5 grid of temperature readings (25 sensors in a grid)
grid = np.arange(1, 26).reshape((5, 5))
grid

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25]])

In [41]:
grid[0]          # first row

array([1, 2, 3, 4, 5])

In [42]:
grid[0, :]       # same - first row, all columns

array([1, 2, 3, 4, 5])

In [43]:
grid[:, 2]       # all rows, third column

array([ 3,  8, 13, 18, 23])

In [44]:
grid[-1]         # last row

array([21, 22, 23, 24, 25])

In [45]:
grid[:3, :3]     # upper-left 3x3 block

array([[ 1,  2,  3],
       [ 6,  7,  8],
       [11, 12, 13]])

In [46]:
grid[3:, 3:]     # lower-right 2x2 corner

array([[19, 20],
       [24, 25]])

In [47]:
grid[::2]        # every other row (rows 0, 2, 4)

array([[ 1,  2,  3,  4,  5],
       [11, 12, 13, 14, 15],
       [21, 22, 23, 24, 25]])

In [48]:
grid[::2, ::2]   # every other row and column

array([[ 1,  3,  5],
       [11, 13, 15],
       [21, 23, 25]])

### Copies vs. views

Slicing an array returns a **view**, not a copy. Modifying the slice modifies the original array.

In [49]:
original = np.array([10, 20, 30, 40, 50])
view = original[1:4]    # this is a view into original
view

array([20, 30, 40])

In [50]:
view[0] = 999
print("original:", original)    # original is also changed

original: [ 10 999  30  40  50]


In [51]:
# Use .copy() when you need an independent array
original = np.array([10, 20, 30, 40, 50])
snapshot = original[1:4].copy()
snapshot

array([20, 30, 40])

In [52]:
snapshot[0] = 999
print("original:", original)    # original is unchanged

original: [10 20 30 40 50]


### Boolean indexing

Pass a boolean array as an index to select elements where the condition is `True`. This is one of the most frequently used NumPy patterns.

In [53]:
# The scores matrix was:
scores

array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84]])

In [54]:
# Which students scored below 70 on any exam?
mask = scores < 70
mask    # a boolean array with the same shape as scores

array([[False, False, False],
       [False, False, False],
       [False,  True, False],
       [False, False, False]])

In [55]:
scores[mask]    # returns a 1D array of all values where condition is True

array([65])

In [56]:
# Find scores above 90
scores[scores > 90]

array([92, 95, 91])

#### Boolean operators in NumPy:

- `&` means perform logical AND
- `|` means logical OR
- `~` means logical NOT

When using the `&`, `|` or the `~` operators in NumpPy, the conditions must be put inside brackets like the following example:

In [57]:
# Compound conditions use & (and) and | (or), each condition in parentheses
scores[(scores >= 80) & (scores <= 90)]    # scores in the 80-90 band

array([85, 90, 88, 88, 84])

### `np.where()`

`np.where(condition)` (single argument, only speciffying a condition) returns the indices where condition is True:

In [58]:
# Which exam scores are below 70?
np.where(scores < 70)

(array([2]), array([1]))

In [59]:
# Which exam scores are below 71?
np.where(scores < 71)

(array([2, 2]), array([0, 1]))

In the above output, the there are two arrays, the first array indicates row indices and the second array indicates column indices. Therefore, the value at (row index 2, column index 0) as well as the value at (row index 2, column index 1) are below 71.

In [60]:
# Which rows (students) have at least one score below 70?
low_rows = np.any(scores < 70, axis=1)
print("Students with a score below 70:", np.where(low_rows)[0])    # row indices

Students with a score below 70: [2]


In [61]:
# On which days temperature was above 30 degrees?
hot_days = np.where(temps > 30)
print("Indices of hot days:", hot_days[0])
print("Temperatures on those days:", temps[hot_days])

Indices of hot days: [1 5 6]
Temperatures on those days: [31.2 33.1 32.6]


`np.where(condition, x, y)` returns `x` where condition is `True` and `y` where it is `False`. It is a vectorized alternative to a loop with an if/else inside.

In [62]:
# Classify each score as Pass or Fail
labels = np.where(scores >= 70, "Pass", "Fail")
labels

array([['Pass', 'Pass', 'Pass'],
       ['Pass', 'Pass', 'Pass'],
       ['Pass', 'Fail', 'Pass'],
       ['Pass', 'Pass', 'Pass']], dtype='<U4')

In [63]:
print("Original temps:", temps)

# Cap all temperatures above 32 at exactly 32
capped = np.where(temps > 32, 32, temps)
print("Capped temps:", capped)

Original temps: [28.5 31.2 30.  27.8 29.4 33.1 32.6]
Capped temps: [28.5 31.2 30.  27.8 29.4 32.  32. ]


## Basic Matrix and Linear Algebra Operations

### Matrix multiplication with `np.matmul()`

In [64]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

B = np.array([
    [9, 8, 7],
    [6, 5, 4],
    [3, 2, 1]
])

np.matmul(A, B)

array([[ 30,  24,  18],
       [ 84,  69,  54],
       [138, 114,  90]])

In [65]:
# Multiplying a matrix by a vector gives a 1D output
v = np.array([1, 0, -1])
print("The shape of v is", v.shape)
np.matmul(A, v)    # v is interpreted as a matrix with three rows and one column

The shape of v is (3,)


array([-2, -2, -2])

In [66]:
np.matmul(v, A)    # v is interpreted as a matrix with three columns and one row

array([-6, -6, -6])

In [67]:
# The @ operator is shorthand for matmul
A @ B

array([[ 30,  24,  18],
       [ 84,  69,  54],
       [138, 114,  90]])

### Trace

In [68]:
# Sum of the diagonal elements
A.trace()    # 1 + 5 + 9 = 15

np.int64(15)

### Rank

In [69]:
np.linalg.matrix_rank(A)    # A is rank-deficient: row 3 = row 1 + row 2, approximately

np.int64(2)

In [70]:
# A full-rank matrix
C = np.array([[2, 5, 2], [1, 4, 3], [7, 2, 5]])
print(C)
print(np.linalg.matrix_rank(C))    # The rank of C is 3

[[2 5 2]
 [1 4 3]
 [7 2 5]]
3


### Inverse

In [71]:
C_inv = np.linalg.inv(C)
C_inv

array([[ 0.25      , -0.375     ,  0.125     ],
       [ 0.28571429, -0.07142857, -0.07142857],
       [-0.46428571,  0.55357143,  0.05357143]])

In [72]:
# Verify: C @ C_inv should be the identity matrix
idt = C @ C_inv
idt

array([[ 1.00000000e+00, -2.22044605e-16,  2.77555756e-17],
       [-2.77555756e-16,  1.00000000e+00,  2.77555756e-17],
       [-6.10622664e-16, -2.22044605e-16,  1.00000000e+00]])

The result is not exactly the identity matrix because floating-point arithmetic has finite precision.

Numpy can also perform many other linear algebra operations such as matrix decomposition (SVD, QR etc) as well as find eigen values, eigen vectors etc.

## Comparing Arrays

### Element-wise comparison

Standard comparison operators return boolean arrays:

In [73]:
week1 == week2    # element-wise equality check

array([False, False, False, False, False, False, False])

In [74]:
week1 > week2

array([ True,  True, False, False, False,  True,  True])

### The problem with `==` for floating-point arrays

Due to the way floating-point numbers are stored in binary, arithmetic operations can introduce tiny rounding errors. Using `==` to compare two arrays that should be equal can return `False`:

In [75]:
# These should be equal, but floating-point arithmetic introduces error
a = np.array([0.1 + 0.2])
b = np.array([0.3])

print("Are they equal? ", a == b)        # False - not reliable for floats
print("Values: ", a[0], b[0])

Are they equal?  [False]
Values:  0.30000000000000004 0.3


### `np.allclose()`

`np.allclose(a, b, rtol, atol)` checks whether two arrays are equal within a tolerance. Use this instead of `==` when comparing floats.

In [76]:
print(np.allclose(a, b))    # True - they are close enough

# Verify that C @ C_inv is actually the identity matrix
identity = np.identity(3)
print(np.allclose(C @ C_inv, identity))    # True

True
True


In [77]:
# You can control the tolerance
np.allclose(a, b, atol=1e-8)    # absolute tolerance of 1e-8

True

## Universal Functions (ufuncs)

A universal function (`ufunc`) applies an operation element-wise to every value in an array. These are implemented in compiled C code and are much faster than equivalent Python loops.

**Unary ufuncs** take one array:

| Function | Description |
|---|---|
| `np.abs()` | Absolute value |
| `np.sqrt()` | Square root |
| `np.square()` | Square (equivalent to `arr ** 2`) |
| `np.exp()` | Exponent $e^x$ |
| `np.log()` | Natural logarithm |
| `np.log10()` | Log base 10 |
| `np.log2()` | Log base 2 |
| `np.sign()` | Sign: 1, 0, or -1 |
| `np.ceil()` | Round up to nearest integer |
| `np.floor()` | Round down to nearest integer |
| `np.isnan()` | Boolean array: is each element NaN? |
| `np.sin()`, `np.cos()`, `np.tan()` | Trigonometric functions |
| `arccos()`, `arcsin()`, `arctan()` | Inverse trigonometric functions|
| `isfinite()`, `isinf()`                                   | Return Boolean array indicating whether each element is finite (non-inf, non-NaN) or infinite, respectively |
| `logical_not()` | Compute truth value of not x element-wise (equivalent to ~arr)                                              |

In [78]:
arr = np.arange(1, 10)
np.sqrt(arr).round(4)

array([1.    , 1.4142, 1.7321, 2.    , 2.2361, 2.4495, 2.6458, 2.8284,
       3.    ])

In [79]:
np.log(arr).round(4)

array([0.    , 0.6931, 1.0986, 1.3863, 1.6094, 1.7918, 1.9459, 2.0794,
       2.1972])

In [80]:
# isnan is useful for detecting missing data
sensor_data = np.array([1.2, np.nan, 3.4, np.nan, 5.6])
np.isnan(sensor_data)

array([False,  True, False,  True, False])

**Binary ufuncs** take two arrays:

| Function | Description |
|---|---|
| `np.add()` | Element-wise addition |
| `np.subtract()` | Element-wise subtraction |
| `np.multiply()` | Element-wise multiplication |
| `np.divide()` | Element-wise division |
| `np.power()` | Raise elements of first array to powers in second |
| `np.maximum()` | Element-wise maximum |
| `np.minimum()` | Element-wise minimum |
| `np.mod()` | Element-wise modulus |

In [81]:
# Element-wise maximum: the higher reading from either station, per day
np.maximum(week1, week2)

array([28.5, 31.2, 31.4, 28.9, 30.2, 33.1, 32.6])

In [82]:
# Raise each score to a custom power per exam (unusual, but demonstrates the function)
exponents = np.array([1, 2, 0.5])
np.power(scores[0], exponents)

array([  85.        , 8100.        ,    8.83176087])

## Statistical and Mathematical Methods

These are accessible as methods on an array or as `np.function(array)`. The `axis` argument controls the direction of reduction:
- `axis=0`: reduce across rows (result has one value per column)
- `axis=1`: reduce across columns (result has one value per row)

| Method | Description |
|---|---|
| `.sum()` | Sum of elements |
| `.mean()` | Arithmetic mean |
| `.std()` | Standard deviation |
| `.var()` | Variance |
| `.min()`, `.max()` | Minimum and maximum |
| `.argmin()`, `.argmax()` | Index of minimum and maximum |
| `.cumsum()` | Cumulative sum |
| `.cumprod()` | Cumulative product |

In [83]:
# Overall average exam score
scores.mean()

np.float64(83.16666666666667)

In [84]:
# Average score per exam (axis=0 collapses the student dimension)
scores.mean(axis=0)    # one mean per exam

array([83.75, 83.5 , 82.25])

In [85]:
# Average score per student (axis=1 collapses the exam dimension)
scores.mean(axis=1)    # one mean per student

array([84.33333333, 91.66666667, 69.        , 87.66666667])

In [86]:
scores.std(axis=0)    # spread of scores on each exam

array([ 8.31790238, 10.73545528,  8.49632273])

In [87]:
# Which exam had the lowest average?
scores.mean(axis=0).argmin()    # returns column index

np.int64(2)

In [88]:
# Cumulative sum of temperatures across the week
temps.cumsum()

array([ 28.5,  59.7,  89.7, 117.5, 146.9, 180. , 212.6])

### Applying arbitrary functions with `np.apply_along_axis()`

`np.apply_along_axis(func, axis, arr)` applies any function to 1D slices of an array. It is most useful when the operation you need is not already available as a built-in method.

`axis=0` means the function receives each **column** as a 1D array (the function is applied along the row axis).
`axis=1` means the function receives each **row** as a 1D array (the function is applied along the column axis).

In [89]:
# Range (max - min) of scores per exam - no built-in method for this
np.apply_along_axis(lambda col: col.max() - col.min(), axis=0, arr=scores)

array([22, 26, 23])

In [90]:
# Same idea per student
np.apply_along_axis(lambda row: row.max() - row.min(), axis=1, arr=scores)

array([12,  7,  7,  7])

## Random Number Generation

NumPy has two APIs for random number generation. The **legacy API** (`np.random.*`) is still widely seen in older code. The **new API** (`np.random.default_rng()`) is preferred for new code because the generator object is explicit and reproducible without relying on global state.

### Legacy API (still common in existing codebases)

In [91]:
np.random.seed(42)
np.random.random(5)    # 5 uniform floats in [0, 1)

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864])

In [92]:
np.random.seed(42)
np.random.standard_normal(size=(2, 4))    # from N(0, 1)

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986],
       [-0.23415337, -0.23413696,  1.57921282,  0.76743473]])

In [93]:
np.random.seed(42)
np.random.normal(loc=70, scale=10, size=10)    # exam-score-like values, mean=70, sd=10

array([74.96714153, 68.61735699, 76.47688538, 85.23029856, 67.65846625,
       67.65863043, 85.79212816, 77.67434729, 65.30525614, 75.42560044])

### New API: `np.random.default_rng()` (preferred)

Create a Generator object with a fixed seed.

In [94]:
rng = np.random.default_rng(seed=42)
rng.standard_normal(size=5)

array([ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519])

In [95]:
# Reproducible: same seed gives the same draws
rng2 = np.random.default_rng(seed=42)
rng2.standard_normal(size=5)    # identical to the block above

array([ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519])

In [96]:
# Simulate student exam scores: mean 72, sd 12, 30 students, 3 exams
rng3 = np.random.default_rng(seed=0)
simulated_scores = rng3.normal(loc=72, scale=12, size=(30, 3)).clip(0, 100).round(1)
simulated_scores[:5]    # first 5 students

array([[73.5, 70.4, 79.7],
       [73.3, 65.6, 76.3],
       [87.6, 83.4, 63.6],
       [56.8, 64.5, 72.5],
       [44.1, 69.4, 57. ]])

Common distributions available on a Generator object:

| Method | Description |
|---|---|
| `.random()` | Uniform floats in [0, 1) |
| `.integers(low, high)` | Random integers in [low, high) |
| `.standard_normal()` | Normal distribution, mean 0, sd 1 |
| `.normal(loc, scale)` | Normal distribution with specified mean and sd |
| `.uniform(low, high)` | Uniform distribution over [low, high) |
| `.binomial(n, p)` | Binomial distribution |
| `.beta(a, b)` | Beta distribution |
| `.gamma(shape, scale)` | Gamma distribution |

Run `dir(rng)` to see the full list.

## Stacking Arrays

### `np.stack()`

Joins a sequence of arrays along a **new** axis. The arrays must have identical shapes.

In [97]:
# Two weeks of exam scores, same shape (4, 3)
week1_scores = np.array([[85, 90, 78], [92, 88, 95], [70, 65, 72], [88, 91, 84]])
week2_scores = np.array([[80, 85, 82], [88, 90, 91], [74, 70, 76], [85, 88, 87]])

stacked = np.stack((week1_scores, week2_scores))
print("Shape:", stacked.shape)    # (2, 4, 3) - a new first axis for the two weeks
stacked

Shape: (2, 4, 3)


array([[[85, 90, 78],
        [92, 88, 95],
        [70, 65, 72],
        [88, 91, 84]],

       [[80, 85, 82],
        [88, 90, 91],
        [74, 70, 76],
        [85, 88, 87]]])

### `np.hstack()` and `np.vstack()`

In [98]:
# vstack: stack along axis 0 (add rows)
combined_students = np.vstack((week1_scores, week2_scores))
print("vstack shape:", combined_students.shape)    # (8, 3)
combined_students

vstack shape: (8, 3)


array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84],
       [80, 85, 82],
       [88, 90, 91],
       [74, 70, 76],
       [85, 88, 87]])

In [99]:
# hstack: stack along axis 1 (add columns)
combined_exams = np.hstack((week1_scores, week2_scores))
print("hstack shape:", combined_exams.shape)    # (4, 6)
combined_exams

hstack shape: (4, 6)


array([[85, 90, 78, 80, 85, 82],
       [92, 88, 95, 88, 90, 91],
       [70, 65, 72, 74, 70, 76],
       [88, 91, 84, 85, 88, 87]])

## Splitting Arrays

`numpy.split(ary, indices_or_sections, axis=0)`

Splits an array into multiple sub-arrays.

**Parameters:**
- `ary`: the array to split
- `indices_or_sections`:
  - If an **integer** N: splits into N equal parts along the given axis. Raises an error if the split is not even.
  - If a **1D array of sorted integers**: the integers are the indices at which to cut. For example, `[2, 5]` on `axis=0` produces `ary[:2]`, `ary[2:5]`, `ary[5:]`.
- `axis`: the axis along which to split (default 0)

In [100]:
sensor_grid = np.arange(1, 25).reshape((4, 6))
sensor_grid

array([[ 1,  2,  3,  4,  5,  6],
       [ 7,  8,  9, 10, 11, 12],
       [13, 14, 15, 16, 17, 18],
       [19, 20, 21, 22, 23, 24]])

In [101]:
# Split into 4 equal row-slices (one row each)
parts = np.split(sensor_grid, 4)
for i, part in enumerate(parts):
    print(f"Part {i}: {part}")

Part 0: [[1 2 3 4 5 6]]
Part 1: [[ 7  8  9 10 11 12]]
Part 2: [[13 14 15 16 17 18]]
Part 3: [[19 20 21 22 23 24]]


In [102]:
# Split at column index 2 and 4 along axis=1
left, middle, right = np.split(sensor_grid, [2, 4], axis=1)
print("Left (cols 0-1):\n",   left)
print("Middle (cols 2-3):\n", middle)
print("Right (cols 4-5):\n",  right)

Left (cols 0-1):
 [[ 1  2]
 [ 7  8]
 [13 14]
 [19 20]]
Middle (cols 2-3):
 [[ 3  4]
 [ 9 10]
 [15 16]
 [21 22]]
Right (cols 4-5):
 [[ 5  6]
 [11 12]
 [17 18]
 [23 24]]


## Saving and Loading Arrays

### Binary format: `np.save()` and `np.load()`

`.npy` is NumPy's native binary format. It preserves dtype and shape exactly and is fast to read and write. Use this when the data will only be consumed by Python/NumPy.

In [103]:
np.save("scores.npy", scores)    # writes scores.npy to the current directory

In [104]:
loaded = np.load("scores.npy")
print(loaded)
print("dtype:", loaded.dtype)
print("shape:", loaded.shape)

[[85 90 78]
 [92 88 95]
 [70 65 72]
 [88 91 84]]
dtype: int64
shape: (4, 3)


### Multiple arrays: `np.savez()` and `np.load()`

`np.savez()` packs multiple arrays into a single `.npz` file (a zip archive of `.npy` files).

In [105]:
np.savez("exam_data.npz", week1=week1_scores, week2=week2_scores)

In [106]:
data = np.load("exam_data.npz")
print("Keys:", list(data.keys()))
print("week1:\n", data["week1"])

Keys: ['week1', 'week2']
week1:
 [[85 90 78]
 [92 88 95]
 [70 65 72]
 [88 91 84]]


### Text format: `np.savetxt()` and `np.loadtxt()`

Use this when the data needs to be human-readable or shared with tools outside Python (spreadsheets, other languages).

In [107]:
np.savetxt("scores.csv", scores, delimiter=",", fmt="%d", header="Exam1,Exam2,Exam3", comments="")

In [108]:
loaded_csv = np.loadtxt("scores.csv", delimiter=",", skiprows=1, dtype=int)
loaded_csv

array([[85, 90, 78],
       [92, 88, 95],
       [70, 65, 72],
       [88, 91, 84]])

**When to use each format:**

- `.npy` / `.npz`: fast, lossless, preserves dtype. Use for intermediate data that stays in Python.
- `.csv` (via `savetxt`): human-readable, portable. Use when sharing with other tools, but be aware that floats may lose precision due to text formatting.

---

## Exercises

**Exercise 1**

Create a 5x5 identity matrix using `np.identity()`. Replace the main diagonal using the `np.fill_diagonal()` function with the values 1 through 5. Print the result and confirm its dtype. You can check the Numpy documentation to see how the `np.fill_diagonal()` function is used.

In [109]:
# Type your solution here:


**Exercise 2**

Create a 1D array of 20 evenly spaced values between 0 and 1 using `np.linspace()`. Reshape it into a 4x5 matrix. Compute the row-wise mean and the column-wise standard deviation.

In [110]:
# Type your solution here:


**Exercise 3**

Create two 3x3 matrices of your choice. Compute their matrix product using `@`. Verify the result for the first row of the output, by computing the dot products manually as scalar arithmetic in a separate cell.

In [111]:
# Type your solution here:


**Exercise 4**

Create a 10x10 array of integers from 1 to 100 using `np.arange()` and `reshape()`. Make a copy of the array. In the copy, replace all values less than 25 with 0 using boolean indexing. Print the original and the modified array to confirm the original is unchanged.

In [112]:
# Type your solution here:


**Exercise 5**

Write a function `standardize(arr)` that takes a 2D array and returns a new array where each row has mean 0 and standard deviation 1 (z-score normalization). Apply it to a randomly generated 5x4 matrix using `np.random.default_rng(seed=7)`. Verify that the row means of the output are all close to 0 using `np.allclose()`.

In [113]:
# Type your solution here:


**Exercise 6**

Using `week1_scores` and `week2_scores` from the stacking section, stack them horizontally and vertically. Print the shape of each result. Then split the horizontally stacked array back into two equal parts along axis 1 and confirm the parts match the originals using `np.allclose()`.

In [114]:
# Type your solution here:
